In [ ]:
# Imports & Path
# Run once at the start of every session

import sys
import os
import time
import random
import datetime

# Add Slurrybot folder to path so Python finds all files
sys.path.append(r"C:\Users\digic\OneDrive\Desktop\SlurryBotPc\Slurry Formation 2024\SlurryBot")

# Load all robot functions from the wrapper
%reload_ext autoreload
%autoreload 2
from Robot_wrapper import *

In [ ]:
# Connect Robot
# Run once at the start of every session

# Connect to robot arm
robot = Robot()
robot.initialize()
print("✓ Robot connected")

In [ ]:
robot.initialize()
robot.GoTo_InitialPoint()
robot.restart()

In [ ]:
# Connect Scale
# Run once at the start of every session

# Connect to scale
from Drivers.scale_driver import scale
scale = Scale('COM4', 9600, 10)
scale.connect()
print("✓ Scale connected")

# scale object methods:
# scale.tare()            → zero the scale
# scale.measure()         → single measurement
# scale.measure_stable()  → measure and wait until value is stable

In [ ]:
# Test Scale
lines = scale.get(CMD_PRINT)
print("Raw response from scale:", lines)

## test scales before use, True = ready
reading = scale.measure()
print(reading.value, reading.stable)

In [ ]:
import time
import random
import datetime

MAX_LARGE_SCOOPS  = 20
MAX_MEDIUM_SCOOPS = 30

log = []

def log_event(event_type, message, weight=None, added=None):
    entry = {
        "time":    datetime.datetime.now().strftime("%H:%M:%S"),
        "type":    event_type,
        "message": message,
        "weight":  weight,
        "added":   added,
    }
    log.append(entry)
    weight_str = f" | total: {weight:.4f}g" if weight is not None else ""
    added_str  = f" | added: +{added:.4f}g" if added  is not None else ""
    print(f"[{entry['time']}] [{event_type.upper():7s}] {message}{added_str}{weight_str}")

def print_run_summary():
    print("\n" + "="*60)
    print("RUN SUMMARY")
    print("="*60)
    print(f"{'Time':<10} {'Type':<10} {'Event':<25} {'Added':>8} {'Total':>8}")
    print("-"*60)
    for entry in log:
        added_str  = f"{entry['added']:>8.4f}g"  if entry['added']  is not None else f"{'':>9}"
        weight_str = f"{entry['weight']:>8.4f}g" if entry['weight'] is not None else f"{'':>9}"
        print(f"{entry['time']:<10} {entry['type'].upper():<10} "
              f"{entry['message']:<25} {added_str} {weight_str}")
    print("="*60)
    scoops = [e for e in log if e["type"] == "scoop"]
    if scoops:
        total = sum(e["added"] for e in scoops if e["added"] is not None)
        print(f"Total scoops:    {len(scoops)}")
        print(f"Total dispensed: {total:.4f}g")
    print("="*60)

def calc_weight_needed(weight_dispensed, target_weight):
    return target_weight - weight_dispensed

def wait_for_stable_weight(previous_weight, max_attempts=30):
    for _ in range(max_attempts):
        raw = scale.measure()
        if raw.stable and abs(raw.value - previous_weight) > 0.01:
            return raw.value
        time.sleep(0.1)
    raise TimeoutError("Scale did not stabilise – check scale connection!")

In [ ]:
# Calibration
# Run before first weighing session or when changing powder
# Compares your results to Aidan's reference values

def calibrate_scoop(scoop_function, name, repetitions=3):
    print(f"\n--- Calibrating: {name} ({repetitions}x) ---")
    scale.tare()
    results        = []
    previous_weight = 0

    for i in range(repetitions):
        scoop_function()
        weight  = wait_for_stable_weight(previous_weight)
        added   = weight - previous_weight
        results.append(added)
        print(f"  Scoop {i+1}: +{added:.4f}g (total: {weight:.4f}g)")
        previous_weight = weight

    avg = sum(results) / len(results)
    print(f"  → Average: {avg:.4f}g | Min: {min(results):.4f}g | Max: {max(results):.4f}g")
    return avg

# Uncomment what you want to calibrate:
# calibrate_scoop(robot.ScoopBigMiddle,    "big middle",     repetitions=3)
# calibrate_scoop(robot.ScoopBigLeft,      "big left",       repetitions=3)
# calibrate_scoop(robot.ScoopBigRight,     "big right",      repetitions=3)
# calibrate_scoop(robot.ScoopMediumLeft,   "medium left",    repetitions=3)
# calibrate_scoop(robot.ScoopMediumMiddle, "medium middle",  repetitions=3)
# calibrate_scoop(robot.ScoopMediumRight,  "medium right",   repetitions=3)
# calibrate_scoop(robot.ScoopMediumFarleft,"medium farleft", repetitions=3)
# calibrate_scoop(robot.ScoopTiny,         "tiny middle",    repetitions=3)
# calibrate_scoop(robot.ScoopTinyLeft,     "tiny left",      repetitions=3)
# calibrate_scoop(robot.ScoopTinyRight,    "tiny right",     repetitions=3)

In [ ]:
# Weighing Logic
# Internal function – called by run_weighing_workflow()

def run_weighing_logic(target_weight):
    global previous_weight, last_scoop
    previous_weight = 0
    last_scoop      = None

    if target_weight >= 1.0:
        robot.PickupLargeSpatula()
        log_event("info", "Large spatula selected")
    else:
        robot.PickupMediumSpatula()
        log_event("info", "Medium spatula selected")

    weight_dispensed = scale.measure_stable().value
    weight_needed    = calc_weight_needed(weight_dispensed, target_weight)
    log_event("info", f"Start: {weight_dispensed}g | Needed: {weight_needed}g")

    if target_weight >= 1.0:
        large_scoop_count = 0

        if 1.0 <= target_weight < 1.5:
            robot.ScoopBigRight()
            last_scoop = robot.ScoopBigRight
            log_event("scoop", "big right (first)")
        elif 1.5 <= target_weight < 2.0:
            robot.ScoopBigLeft()
            last_scoop = robot.ScoopBigLeft
            log_event("scoop", "big left (first)")
        else:
            robot.ScoopBigMiddle()
            last_scoop = robot.ScoopBigMiddle
            log_event("scoop", "big middle (first)")
        large_scoop_count += 1

        weight_dispensed = wait_for_stable_weight(previous_weight)
        weight_needed    = calc_weight_needed(weight_dispensed, target_weight)
        fraction_left    = weight_needed / target_weight
        log_event("scoop", "first scoop measured",
                  weight=weight_dispensed,
                  added=weight_dispensed - previous_weight)
        previous_weight = weight_dispensed

        while fraction_left > 0.37:
            if large_scoop_count >= MAX_LARGE_SCOOPS:
                log_event("warning", f"Large limit ({MAX_LARGE_SCOOPS}) reached – container empty?")
                print("⚠️ Stopping large loop!")
                break

            if fraction_left > 0.62:
                robot.ScoopBigMiddle()
                last_scoop = robot.ScoopBigMiddle
                log_event("scoop", "big middle")
            elif 0.5 <= fraction_left <= 0.62:
                robot.ScoopBigLeft()
                last_scoop = robot.ScoopBigLeft
                log_event("scoop", "big left")
            elif 0.4 <= fraction_left < 0.5:
                last_scoop()
                log_event("scoop", "repeat last scoop")
            elif 0.37 <= fraction_left < 0.4:
                robot.ScoopBigRight()
                last_scoop = robot.ScoopBigRight
                log_event("scoop", "big right")
            else:
                log_event("error", "Unexpected state!")
                break

            large_scoop_count += 1
            weight_dispensed = wait_for_stable_weight(previous_weight)
            weight_needed    = calc_weight_needed(weight_dispensed, target_weight)
            fraction_left    = weight_needed / target_weight
            log_event("scoop", f"large #{large_scoop_count}",
                      weight=weight_dispensed,
                      added=weight_dispensed - previous_weight)
            previous_weight = weight_dispensed

        if 0.03 <= fraction_left < 0.37:
            log_event("info", "Switching to medium spatula")
            robot.ReplaceLargeSpatula()
            robot.PickupMediumSpatula()

            medium_scoop_count = 0

            while weight_needed > 0.05:
                if medium_scoop_count >= MAX_MEDIUM_SCOOPS:
                    log_event("warning", f"Medium limit ({MAX_MEDIUM_SCOOPS}) reached – container empty?")
                    print("⚠️ Stopping medium loop!")
                    break

                if weight_needed >= 0.6:
                    robot.ScoopMediumLeft()
                    log_event("scoop", "medium left")
                elif 0.4 <= weight_needed < 0.6:
                    robot.ScoopMediumMiddle()
                    log_event("scoop", "medium middle")
                elif 0.3 <= weight_needed < 0.4:
                    robot.ScoopMediumRight()
                    log_event("scoop", "medium right")
                elif 0.2 <= weight_needed < 0.3:
                    robot.ScoopMediumFarleft()
                    log_event("scoop", "medium farleft")
                elif 0.08 <= weight_needed < 0.2:
                    robot.ScoopTinyRight()
                    log_event("scoop", "tiny right")
                elif 0.04 <= weight_needed < 0.08:
                    robot.ScoopTinyLeft()
                    log_event("scoop", "tiny left")
                else:
                    log_event("warning", "Remainder too small – manual adjustment")
                    print("⚠️ Remainder too small")
                    break

                medium_scoop_count += 1
                weight_dispensed = wait_for_stable_weight(previous_weight)
                weight_needed    = calc_weight_needed(weight_dispensed, target_weight)
                log_event("scoop", f"medium #{medium_scoop_count}",
                          weight=weight_dispensed,
                          added=weight_dispensed - previous_weight)
                previous_weight = weight_dispensed

            difference = target_weight - weight_dispensed
            if difference < -0.02:
                log_event("result", f"OVERSHOT by {abs(difference):.4f}g", weight=weight_dispensed)
                print(f"⚠️ Overshot! {weight_dispensed:.4f}g (by {abs(difference):.4f}g)")
            elif -0.02 <= difference <= 0.02:
                log_event("result", f"TARGET REACHED, off by {abs(difference):.4f}g", weight=weight_dispensed)
                print(f"✅ Done! {weight_dispensed:.4f}g (off by {abs(difference):.4f}g)")
            else:
                log_event("result", f"UNDERSHOT by {difference:.4f}g", weight=weight_dispensed)
                print(f"❓ Undershot. {weight_dispensed:.4f}g ({difference:.4f}g still needed)")

            robot.ReplaceMediumSpatula()
        else:
            log_event("error", f"Fraction {fraction_left:.4f} outside expected range")
            print("Error: fraction outside expected range")

    else:
        medium_scoop_count = 0

        while weight_needed > 0.02:
            if medium_scoop_count >= MAX_MEDIUM_SCOOPS:
                log_event("warning", f"Medium limit ({MAX_MEDIUM_SCOOPS}) reached – container empty?")
                print("⚠️ Stopping!")
                break

            if weight_needed >= 0.6:
                robot.ScoopMediumLeft()
                log_event("scoop", "medium left")
            elif 0.4 <= weight_needed < 0.6:
                robot.ScoopMediumMiddle()
                log_event("scoop", "medium middle")
            elif 0.3 <= weight_needed < 0.4:
                robot.ScoopMediumRight()
                log_event("scoop", "medium right")
            elif 0.2 <= weight_needed < 0.3:
                robot.ScoopMediumFarleft()
                log_event("scoop", "medium farleft")
            elif 0.08 <= weight_needed < 0.2:
                robot.ScoopTiny()
                log_event("scoop", "tiny middle")
            elif 0.02 < weight_needed < 0.08:
                scoop_fn = random.choice([robot.ScoopTinyLeft, robot.ScoopTinyRight])
                log_event("scoop", f"tiny random ({scoop_fn.__name__})")
                scoop_fn()
            else:
                log_event("warning", "Remainder too small – manual adjustment")
                break

            medium_scoop_count += 1
            weight_dispensed = wait_for_stable_weight(previous_weight)
            weight_needed    = calc_weight_needed(weight_dispensed, target_weight)
            log_event("scoop", f"medium #{medium_scoop_count}",
                      weight=weight_dispensed,
                      added=weight_dispensed - previous_weight)
            previous_weight = weight_dispensed

        difference = target_weight - weight_dispensed
        if difference < -0.02:
            log_event("result", f"OVERSHOT by {abs(difference):.4f}g", weight=weight_dispensed)
            print(f"⚠️ Overshot! {weight_dispensed:.4f}g (by {abs(difference):.4f}g)")
        elif -0.02 <= difference <= 0.02:
            log_event("result", f"TARGET REACHED, off by {abs(difference):.4f}g", weight=weight_dispensed)
            print(f"✅ Done! {weight_dispensed:.4f}g (off by {abs(difference):.4f}g)")
        else:
            log_event("result", f"UNDERSHOT by {difference:.4f}g", weight=weight_dispensed)
            print(f"❓ Undershot. {weight_dispensed:.4f}g ({difference:.4f}g still needed)")

        robot.ReplaceMediumSpatula()

In [ ]:
# Main Workflow

def weighing_workflow(vial_number, target_weight):
    """
    Full workflow:
    1. Pick up vial → place on scale
    2. Place funnel
    3. Weigh powder
    4. Replace funnel
    """
    log.clear()
    log_event("info", f"Starting | Vial: {vial_number} | Target: {target_weight}g")

    # Step 1 – Vial to scale
    log_event("info", f"Picking up {vial_number}")
    robot.PickUpVial(vial_number)
    robot.VialToScale()

    # Step 2 – Funnel
    log_event("info", "Placing funnel")
    robot.PlaceFunnel()

    # Step 3 – Weigh
    scale.tare()
    log_event("info", "Scale tared – starting weighing")
    run_weighing_logic(target_weight)

    # Step 4 – Replace funnel
    log_event("info", "Replacing funnel")
    robot.ReplaceFunnel()

    log_event("info", "⚠️ Remember to manually remove vial from scale!")
    print_run_summary()

In [ ]:
# ── Only weighing (vial already in scale) ─────────────────────────
run_weighing_logic(target_weight=1.5)

In [ ]:
# ── Run full experiment ───────────────────────────────────────────
weighing_workflow(vial_number="Vial1", target_weight=1.5)

In [ ]:
# ── Test einzelne Bewegungen ────────────────────────────────
# Immer nur eine Zelle auf einmal ausführen!

# Grundposition
robot.GoTo_InitialPoint()

In [ ]:
# ── Vial Test ────────────────────────────────────────────────
# Reihenfolge einhalten!
robot.PickUpVial("Vial1")   # 1. Vial holen
robot.VialToScale()          # 2. zur Waage

In [ ]:
# ── Funnel Test ──────────────────────────────────────────────
robot.PlaceFunnel()          # Vial muss auf Waage stehen!
robot.ReplaceFunnel()

In [ ]:
# ── Großer Spatel Test ───────────────────────────────────────
# Komplett als Block ausführen!
robot.PickupLargeSpatula()   # 1. holen
robot.ScoopBigMiddle()       # 2. schaufeln (mehrfach möglich)
robot.ScoopBigLeft()         # optional
robot.ScoopBigRight()        # optional
robot.ReplaceLargeSpatula()  # 3. zurücklegen

In [ ]:
# ── Mittlerer Spatel Test ────────────────────────────────────
robot.PickupMediumSpatula()  # 1. holen
robot.ScoopMediumLeft()      # 2. schaufeln
robot.ScoopMediumMiddle()    # optional
robot.ScoopMediumRight()     # optional
robot.ScoopMediumFarleft()   # optional
robot.ScoopTiny()            # optional
robot.ScoopTinyLeft()        # optional
robot.ScoopTinyRight()       # optional
robot.ReplaceMediumSpatula() # 3. zurücklegen

In [ ]:
# ── Pipette Test ─────────────────────────────────────────────
# Vial muss bei VialRestPoint stehen!
robot.PickUpPipette()        # 1. Pipette holen
robot.GettingPipetteTip("1") # 2. Tip aufstecken
robot.TakingLiquid("1")      # 3. Flüssigkeit aufziehen
robot.LiquidToVial("1")      # 4. in Vial dispensieren
robot.PuttingBackPipetteTip("1")  # 5. Tip abwerfen
robot.PuttingBackPipette()   # 6. Pipette zurück

In [ ]:
# Nur Wiegen testen (Vial schon auf Waage)
scale.tare()
run_weighing_logic(target_weight=1.5)